# Chapter 6 Lab — LoRA Recipe Walkthrough (NVIDIA NeMo-style config)

**Estimated time:** 25 minutes · **Read-and-run.** We parse a NeMo-style LoRA config and
compute how few parameters LoRA actually trains. No YAML library required.

In [ ]:
config_text = '''
model:
  restore_from_path: llama2-7b.nemo
  peft:
    peft_scheme: lora
    lora_tuning:
      adapter_dim: 16          # rank r
      alpha: 32                # scaling (alpha/r)
      target_modules: [attention_qkv, attention_dense]
trainer:
  precision: bf16
'''
print(config_text)

In [ ]:
# Tiny indentation-aware reader (enough for this recipe).
def grab(txt, key):
    for line in txt.splitlines():
        s=line.strip()
        if s.startswith(key+':'):
            return s.split(':',1)[1].split('#')[0].strip()
    return None
r   = int(grab(config_text,'adapter_dim'))
alpha = int(grab(config_text,'alpha'))
mods = grab(config_text,'target_modules')
print(f'rank r          = {r}')
print(f'alpha           = {alpha}  -> scaling alpha/r = {alpha/r:.1f}')
print(f'target_modules  = {mods}')

In [ ]:
# How many params does LoRA train vs full fine-tuning?
d_model=4096; n_layers=32
# LoRA on q,k,v,o projections: each is d x d, replaced by two r-rank matrices (2*d*r).
proj_per_layer=4
lora_params = n_layers*proj_per_layer*(2*d_model*r)
full_params = 7_000_000_000
print(f'LoRA trainable params ~ {lora_params/1e6:.1f} M')
print(f'Full model params     ~ {full_params/1e9:.1f} B')
print(f'LoRA trains {100*lora_params/full_params:.3f}% of the weights')

## Key takeaway
LoRA/QLoRA give **efficiency** (train a fraction of the weights, merge adapters later); DPO
gives **simpler preference alignment** than RLHF (no separate reward model / PPO loop).
Matching technique to data budget and objective is the core skill this domain tests.